# Train Model Nhận diện Nhịp Thở từ WiFi CSI

Notebook này train một mô hình **1D-CNN + LSTM** để ước lượng BPM từ tín hiệu CSI đã qua tiền xử lý.

## Kiến trúc Pipeline
```
CSV Dataset → Tiền xử lý (Hampel, PCA, Band-pass) → Sliding Windows → Train 1D-CNN/LSTM → Export ONNX
```

## Output
- `models/respiration_model.onnx` : Triển khai trên Jetson Nano
- `models/respiration_model.h5`  : Backup Keras model
- `models/training_report.png`   : Báo cáo đánh giá model (4 panel)
- `models/training_history.npz`  : History dùng với `visualize.py`
- `asset/training_report.png`    : Copy sang thư mục asset để nhúng vào README

## 0. Cài đặt thư viện

In [ ]:
# !pip install tensorflow scikit-learn pandas numpy scipy matplotlib seaborn onnx tf2onnx

## 1. Import & Config

In [ ]:
import os, re, sys, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from edge.src.processor import process_window

# ── CONFIG ─────────────────────────────────────────────
DATASET_DIR   = '../datasets'
MODELS_DIR    = '../models'
ASSET_DIR     = '../asset'
SAMPLING_RATE = 100.0
WINDOW_SEC    = 30
WINDOW_SIZE   = int(SAMPLING_RATE * WINDOW_SEC)   # 3000
STEP_SEC      = 5
STEP_SIZE     = int(SAMPLING_RATE * STEP_SEC)      # 500
BPM_MIN, BPM_MAX = 4.0, 40.0
# ────────────────────────────────────────────────────────

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ASSET_DIR, exist_ok=True)
print(f'Config: WINDOW={WINDOW_SEC}s, STEP={STEP_SEC}s, fs={SAMPLING_RATE}Hz')

## 2. Load Dataset

Mỗi file CSV có format: `TenNguoi_BPMxx_Ngay.csv`
Ground Truth BPM được đọc từ tên file.

In [ ]:
def parse_bpm_from_filename(filepath):
    match = re.search(r'BPM(\d+)', os.path.basename(filepath), re.IGNORECASE)
    if match: return float(match.group(1))
    raise ValueError(f'Không tìm thấy BPM: {os.path.basename(filepath)}')

def parse_csi_array(row):
    match = re.search(r'\[([-\d ]+)\]', str(row))
    if not match: return None
    try:
        values = list(map(int, match.group(1).split()))
        return np.array(values) if len(values) >= 64 else None
    except ValueError:
        return None

def load_csv(filepath):
    try:
        df = pd.read_csv(filepath, header=None, on_bad_lines='skip')
    except Exception as e:
        print(f'[!] Lỗi đọc {filepath}: {e}'); return None
    rows = [parse_csi_array(v) for v in df.iloc[:, -1]]
    rows = [r for r in rows if r is not None]
    return np.array(rows) if rows else None

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
print(f'Tìm thấy {len(csv_files)} file CSV:')
for f in csv_files: print(f'  {os.path.basename(f)}')

## 3. Tiền xử lý & Tạo Windows (Sliding Window Data Augmentation)

In [ ]:
X_list, y_list, scenario_list = [], [], []

for filepath in csv_files:
    try:
        bpm_label = parse_bpm_from_filename(filepath)
    except ValueError as e:
        print(f'[SKIP] {e}'); continue

    # Nhãn kịch bản từ tên file
    scenario = 'Normal'
    for tag in ['Slow', 'Fast', 'Apnea', 'Noise']:
        if tag.lower() in os.path.basename(filepath).lower():
            scenario = tag; break

    csi_raw = load_csv(filepath)
    if csi_raw is None or len(csi_raw) < WINDOW_SIZE:
        print(f'[SKIP] {os.path.basename(filepath)}: Không đủ dữ liệu'); continue

    n_windows = 0
    for start in range(0, len(csi_raw) - WINDOW_SIZE + 1, STEP_SIZE):
        window = csi_raw[start: start + WINDOW_SIZE]
        processed = process_window(window, fs=SAMPLING_RATE)
        X_list.append(processed)
        y_list.append(bpm_label)
        scenario_list.append(scenario)
        n_windows += 1

    print(f'[OK] {os.path.basename(filepath)} → BPM={bpm_label} [{scenario}] | {n_windows} windows')

X = np.array(X_list)
y = np.array(y_list)
print(f'\n[Dataset] X={X.shape}, y={y.shape}')
print(f'[Dataset] BPM: min={y.min()}, max={y.max()}, mean={y.mean():.2f}')

## 4. Visualize Dataset

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Dataset Overview', fontsize=13, fontweight='bold')

# BPM Distribution
axes[0].hist(y, bins=15, edgecolor='white', color='#2196F3', alpha=0.85)
axes[0].axvline(12, color='green', linestyle='--', alpha=0.7, label='Normal min')
axes[0].axvline(20, color='orange', linestyle='--', alpha=0.7, label='Normal max')
axes[0].set_title('Ground Truth BPM Distribution')
axes[0].set_xlabel('BPM'); axes[0].set_ylabel('Count'); axes[0].legend()

# Scenario Distribution
cnt = Counter(scenario_list)
colors = {'Normal':'#4CAF50','Slow':'#2196F3','Fast':'#F44336','Apnea':'#9C27B0','Noise':'#FF9800'}
axes[1].bar(cnt.keys(), cnt.values(), color=[colors.get(k,'gray') for k in cnt.keys()], alpha=0.85)
axes[1].set_title('Samples per Scenario'); axes[1].set_ylabel('Count')

# Example breathing signal
t = np.arange(WINDOW_SIZE) / SAMPLING_RATE
axes[2].plot(t, X[0], linewidth=0.8, color='#FF5722')
axes[2].set_title(f'Example Signal (BPM={y[0]:.0f})')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/data_overview.png', dpi=150)
plt.savefig(f'{ASSET_DIR}/data_overview.png', dpi=150)
plt.show()

## 5. Train/Test Split & Chuẩn hóa

In [ ]:
from sklearn.model_selection import train_test_split

X_cnn = X[..., np.newaxis]  # (n, WINDOW_SIZE, 1)
y_norm = (y - BPM_MIN) / (BPM_MAX - BPM_MIN)

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X_cnn, y_norm, scenario_list, test_size=0.2, random_state=42, shuffle=True
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 6. Xây dựng Model: 1D-CNN + LSTM

| Layer | Vai trò |
|:---|:---|
| 1D-CNN Block 1 | Trích xuất đặc trưng nhịp thở cục bộ |
| 1D-CNN Block 2 | Học pattern ở tần số cao hơn |
| LSTM | Nắm bắt xu hướng dài hạn của sóng nhịp thở |
| Dense (sigmoid) | Hồi quy ra BPM [0,1] |

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.random.set_seed(42)

def build_model(input_length):
    inp = keras.Input(shape=(input_length, 1), name='breathing_signal')
    x = layers.Conv1D(32, 50, strides=5, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 25, strides=2, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', name='bpm')(x)
    return keras.Model(inp, out, name='respiration_cnn_lstm')

model = build_model(WINDOW_SIZE)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

## 7. Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(f'{MODELS_DIR}/respiration_model.h5', save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    epochs=100, batch_size=16, validation_split=0.15,
    callbacks=callbacks, verbose=1
)

# Lưu history để dùng với visualize.py --training-history
np.savez(f'{MODELS_DIR}/training_history.npz',
    loss=history.history['loss'],
    val_loss=history.history['val_loss'],
    mae=history.history['mae'],
    val_mae=history.history['val_mae']
)
print('[OK] Đã lưu training_history.npz')

## 8. Đánh giá Model — Training Report (4 Panel)

Hình này sẽ tự động lưu vào `models/training_report.png` và `asset/training_report.png`.

| Panel | Nội dung | Mục tiêu |
|:---|:---|:---|
| A | Loss curve (MSE) | Val Loss hội tụ ổn định |
| B | MAE theo epoch (BPM) | Val MAE ≤ 2.0 BPM |
| C | Predicted vs True scatter | Điểm gần đường y=x |
| D | Error Distribution | Phân phối tập trung quanh 0 |

In [ ]:
best_model = keras.models.load_model(f'{MODELS_DIR}/respiration_model.h5')

y_pred_norm = best_model.predict(X_test).flatten()
y_pred_bpm  = y_pred_norm * (BPM_MAX - BPM_MIN) + BPM_MIN
y_true_bpm  = np.array(y_test) * (BPM_MAX - BPM_MIN) + BPM_MIN
errors      = y_pred_bpm - y_true_bpm
mae_final   = np.mean(np.abs(errors))
rmse_final  = np.sqrt(np.mean(errors**2))
within_2    = np.mean(np.abs(errors) <= 2.0) * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Training Report — 1D-CNN + LSTM', fontsize=14, fontweight='bold')

# A: Loss
axes[0,0].plot(history.history['loss'], '#2196F3', label='Train Loss')
axes[0,0].plot(history.history['val_loss'], '#F44336', label='Val Loss')
best_ep = np.argmin(history.history['val_loss'])
axes[0,0].axvline(best_ep, color='green', linestyle=':', label=f'Best epoch={best_ep}')
axes[0,0].set_title('(A) Loss (MSE)'); axes[0,0].set_xlabel('Epoch')
axes[0,0].legend(); axes[0,0].grid(True, linestyle='--', alpha=0.4)

# B: MAE in BPM
mae_bpm     = np.array(history.history['mae']) * (BPM_MAX - BPM_MIN)
val_mae_bpm = np.array(history.history['val_mae']) * (BPM_MAX - BPM_MIN)
axes[0,1].plot(mae_bpm, '#4CAF50', label='Train MAE')
axes[0,1].plot(val_mae_bpm, '#FF9800', label='Val MAE')
axes[0,1].axhline(2.0, color='red', linestyle='--', linewidth=1.2, label='Target ≤ 2 BPM')
axes[0,1].set_title('(B) MAE (BPM)'); axes[0,1].set_xlabel('Epoch')
axes[0,1].legend(); axes[0,1].grid(True, linestyle='--', alpha=0.4)

# C: Predicted vs True
ZONE_COLORS = {'Normal':'#4CAF50','Slow':'#2196F3','Fast':'#F44336','Apnea':'#9C27B0','Noise':'#FF9800'}
for z in set(s_test):
    idx = [i for i,s in enumerate(s_test) if s==z]
    axes[1,0].scatter(y_true_bpm[idx], y_pred_bpm[idx], label=z,
                      color=ZONE_COLORS.get(z,'gray'), s=60, alpha=0.8, edgecolors='white', lw=0.5)
lim = [0, max(y_true_bpm.max(), y_pred_bpm.max()) + 3]
axes[1,0].plot(lim, lim, 'k--', alpha=0.5, linewidth=1.2)
axes[1,0].fill_between(lim, [l-2 for l in lim], [l+2 for l in lim], alpha=0.08, color='green', label='±2 BPM')
axes[1,0].set_title(f'(C) Predicted vs True\nMAE={mae_final:.2f} | RMSE={rmse_final:.2f}')
axes[1,0].set_xlabel('Ground Truth BPM'); axes[1,0].set_ylabel('Predicted BPM')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, linestyle='--', alpha=0.3)

# D: Error Distribution
axes[1,1].hist(errors, bins=20, color='#9C27B0', edgecolor='white', alpha=0.8)
axes[1,1].axvline(0, color='black', linestyle='--')
axes[1,1].axvline(-2, color='green', linestyle=':'); axes[1,1].axvline(+2, color='green', linestyle=':')
axes[1,1].set_title(f'(D) Error Distribution\n{within_2:.1f}% within ±2 BPM')
axes[1,1].set_xlabel('Error (BPM)'); axes[1,1].set_ylabel('Count')
axes[1,1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/training_report.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{ASSET_DIR}/training_report.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n===== KẾT QUẢ ĐÁNH GIÁ =====')
print(f'MAE  : {mae_final:.2f} BPM')
print(f'RMSE : {rmse_final:.2f} BPM')
print(f'Trong ±2 BPM: {within_2:.1f}%')
print(f'Mục tiêu ≤ 2.0 BPM: {"✓ ĐẠT" if mae_final <= 2.0 else "✗ CHƯA ĐẠT"}')

## 9. So sánh Signal Processing vs Deep Learning

So sánh trực tiếp FFT+Peaks với ONNX Model trên cùng test set.

In [ ]:
sys.path.insert(0, os.path.abspath('../edge'))
from src.estimator import estimate_bpm

y_sp_bpm = []   # Signal Processing
y_dl_bpm = []   # Deep Learning

for i in range(len(X_test)):
    sig = X_test[i, :, 0]  # (WINDOW_SIZE,)
    # Signal Processing
    r = estimate_bpm(sig, fs=SAMPLING_RATE)
    y_sp_bpm.append(r['bpm'])
    # DL
    pred = best_model.predict(X_test[i:i+1], verbose=0)[0][0]
    y_dl_bpm.append(pred * (BPM_MAX - BPM_MIN) + BPM_MIN)

y_sp_bpm = np.array(y_sp_bpm)
y_dl_bpm = np.array(y_dl_bpm)

mae_sp = np.mean(np.abs(y_sp_bpm - y_true_bpm))
mae_dl = np.mean(np.abs(y_dl_bpm - y_true_bpm))
rmse_sp = np.sqrt(np.mean((y_sp_bpm - y_true_bpm)**2))
rmse_dl = np.sqrt(np.mean((y_dl_bpm - y_true_bpm)**2))

# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Signal Processing vs Deep Learning', fontsize=13, fontweight='bold')

methods = ['FFT + Peak\n(Signal Processing)', '1D-CNN + LSTM\n(Deep Learning)']
maes  = [mae_sp, mae_dl]
rmses = [rmse_sp, rmse_dl]
x = np.arange(2)

axes[0].bar(x - 0.2, maes, 0.35, color=['#2196F3', '#F44336'], alpha=0.8, label='MAE')
axes[0].bar(x + 0.2, rmses, 0.35, color=['#2196F3', '#F44336'], alpha=0.4, label='RMSE')
axes[0].set_xticks(x); axes[0].set_xticklabels(methods, fontsize=9)
axes[0].axhline(2.0, color='red', linestyle='--', label='Target 2 BPM')
axes[0].set_title('MAE & RMSE Comparison'); axes[0].set_ylabel('BPM')
axes[0].legend(); axes[0].grid(True, axis='y', linestyle='--', alpha=0.4)
for i, (m, r) in enumerate(zip(maes, rmses)):
    axes[0].text(i-0.2, m+0.05, f'{m:.2f}', ha='center', fontsize=9)

# Scatter comparison overlay
axes[1].scatter(y_true_bpm, y_sp_bpm, color='#2196F3', alpha=0.6, s=40, label=f'FFT (MAE={mae_sp:.2f})')
axes[1].scatter(y_true_bpm, y_dl_bpm, color='#F44336', alpha=0.6, s=40, marker='x', label=f'DL (MAE={mae_dl:.2f})')
lim = [0, max(y_true_bpm.max(), y_sp_bpm.max(), y_dl_bpm.max()) + 3]
axes[1].plot(lim, lim, 'k--', alpha=0.4)
axes[1].set_xlim(lim); axes[1].set_ylim(lim)
axes[1].set_title('Predicted vs True (Both Methods)')
axes[1].set_xlabel('Ground Truth BPM'); axes[1].set_ylabel('Predicted BPM')
axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/method_comparison.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{ASSET_DIR}/method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFFT + Peaks : MAE={mae_sp:.2f} | RMSE={rmse_sp:.2f}')
print(f'1D-CNN+LSTM: MAE={mae_dl:.2f} | RMSE={rmse_dl:.2f}')
better = 'DL' if mae_dl < mae_sp else 'Signal Processing'
print(f'→ {better} tốt hơn trên dataset này.')

## 10. Export ONNX cho Jetson Nano

In [ ]:
import subprocess

onnx_path = f'{MODELS_DIR}/respiration_model.onnx'
result = subprocess.run(
    ['python', '-m', 'tf2onnx.convert',
     '--keras', f'{MODELS_DIR}/respiration_model.h5',
     '--output', onnx_path, '--opset', '13'],
    capture_output=True, text=True
)
print(result.stdout, result.stderr)

if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024**2
    print(f'[OK] Export thành công: {onnx_path} ({size_mb:.2f} MB)')
else:
    print('[FAIL] Kiểm tra lại tf2onnx.')

## 11. Lưu Metadata

In [ ]:
metadata = {
    'bpm_min': BPM_MIN, 'bpm_max': BPM_MAX,
    'window_size': WINDOW_SIZE, 'sampling_rate': SAMPLING_RATE,
    'model_mae_bpm': round(float(mae_final), 2),
    'model_rmse_bpm': round(float(rmse_final), 2),
    'within_2bpm_pct': round(float(within_2), 1),
    'sp_mae_bpm': round(float(mae_sp), 2),
    'sp_rmse_bpm': round(float(rmse_sp), 2)
}
with open(f'{MODELS_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('[OK] Metadata:', json.dumps(metadata, indent=2))